# El radón de Minnesota, en tres modelos

919 mediciones de radón en viviendas de 85 condados de Minnesota. Son los datos de Gelman
y Hill. Llevan veinte años como ejemplo de modelo jerárquico.

El radón es un gas radiactivo. Sale del suelo. La EPA recomienda actuar por encima de
4 pCi/L.

La pregunta: **¿cuánto radón cabe esperar en una casa de un condado concreto?**

El recorrido:

1. Un nivel común para todos los condados, en OLS y en bayesiano.
2. Un nivel propio para cada condado.
3. El jerárquico: los condados comparten información.
4. Diagnóstico y predicción.

No hay ejercicios. Se ejecuta de arriba abajo.

---

**Cómo ejecutarlo.** En Colab, la primera celda instala lo necesario. Los datos se leen
por URL. Si Colab pide reiniciar, reinicia y vuelve a empezar. En local hacen falta
`numpy`, `pandas`, `matplotlib`, `statsmodels`, `pymc` y `arviz`. Los tres modelos tardan
un par de minutos.

In [ ]:
# En Colab: instala lo que falta. En local no hace nada: se da por hecho el entorno.
try:
    import google.colab  # noqa: F401

    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    %pip install -q "numpy>=1.26,<2.5" "pymc>=5.15,<6" "arviz>=0.23,<1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az
import statsmodels.api as sm

ACENTO = "#800080"
GRISES = ["#000000", "#4A4A4A", "#7A7A7A", "#AAAAAA"]

plt.rcParams.update({
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#E6D6E6",
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler("color", GRISES),
    "axes.titlecolor": ACENTO,
    "figure.dpi": 110,
    "font.size": 13,
})

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos

Cada fila es una vivienda. Usamos cuatro columnas:

- `log_radon`: logaritmo de la medición, en pCi/L.
- `floor`: 0 si se midió en el sótano, 1 si en la planta baja.
- `county` y `county_code`: el condado, con nombre y con índice de 0 a 84.

In [ ]:
# El fichero de Gelman, leído del repositorio de ejemplos de PyMC. Fijado al tag
# 2026.02.0: en `main` alguien puede mover el fichero cualquier martes por la tarde.
URL_DATOS = (
    "https://raw.githubusercontent.com/pymc-devs/pymc-examples/"
    "2026.02.0/examples/data/radon.csv"
)

df = pd.read_csv(URL_DATOS)[["county", "county_code", "floor", "log_radon"]]

floor = df["floor"].values
log_radon = df["log_radon"].values
county_idx = df["county_code"].values
n_condados = df["county_code"].nunique()

print(f"{len(df)} mediciones en {n_condados} condados")
df.head()

### El sótano da más radón que la planta baja

El gas sube desde el suelo. Por eso el modelo lleva un coeficiente para la planta.

In [ ]:
# density=True: hay 766 mediciones en sótano y 153 en planta baja.
bordes = np.linspace(-3, 4.5, 25)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(log_radon[floor == 0], bins=bordes, density=True, color="#AAAAAA", label="Sótano")
ax.hist(log_radon[floor == 1], bins=bordes, density=True, histtype="step",
        color=ACENTO, linewidth=2, label="Planta baja")
ax.set_xlabel("log(radón)")
ax.set_ylabel("Densidad")
ax.legend()
plt.show()

## 2. Modelo 1: un nivel común para todos

Una recta, sin condados:

$$\log(\text{radón}_i) = \alpha + \beta_1 \cdot \text{planta}_i + \varepsilon_i$$

Primero, en OLS.

In [ ]:
ols = sm.OLS(df["log_radon"], sm.add_constant(df[["floor"]])).fit()

pd.DataFrame({
    "Coeficiente": ols.params,
    "IC 2,5%": ols.conf_int()[0],
    "IC 97,5%": ols.conf_int()[1],
    "p-valor": ols.pvalues,
}).round(3)

### El mismo modelo, en bayesiano

Ahora hay que escribir las prioris. Sus escalas salen de la variable: `log_radon` tiene
desviación típica 0,82.

- $\alpha \sim \mathcal{N}(0, 2^2)$: el nivel base.
- $\beta_1 \sim \mathcal{N}(0, 1^2)$: el efecto de la planta, sin dirección fijada.
- $\sigma \sim \text{Exp}(1)$: la variación entre viviendas.

In [ ]:
with pm.Model() as modelo_agrupado:
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    beta1 = pm.Normal("beta1", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    y_obs = pm.Normal("y_obs", mu=alpha + beta1 * floor, sigma=sigma, observed=log_radon)

### Antes de ajustar, simulamos desde la priori

Generamos radón sin mirar los datos. Lo pasamos a pCi/L. Comprobamos que es creíble.

In [ ]:
def pintar_previa(previa):
    # Histograma del radón simulado desde la priori, en pCi/L.
    radon = np.exp(previa.prior_predictive["y_obs"].values.reshape(-1))
    p50, p90 = np.percentile(radon, [50, 90])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(np.log10(radon), bins=np.linspace(-6, 10, 65), color=ACENTO)
    ax.axvline(np.log10(4), color="black", linestyle="--", linewidth=1)
    ax.text(np.log10(4) + 0.2, ax.get_ylim()[1] * 0.85, "límite EPA\n(4 pCi/L)", fontsize=10)
    ax.set_xlabel("log₁₀ del radón simulado (pCi/L)")
    ax.set_yticks([])
    ax.set_title(f"Mediana {p50:.3g} pCi/L · percentil 90 {p90:.3g} pCi/L")
    plt.show()


with modelo_agrupado:
    previa_agrupado = pm.sample_prior_predictive(draws=500, random_seed=SEMILLA)

pintar_previa(previa_agrupado)

La mediana queda por debajo de 1 pCi/L. El percentil 90 ronda los 20. La cola es gorda,
pero los órdenes de magnitud son plausibles.

**Pruébalo tú.** Cambia `sigma=2` por `sigma=10` en el `alpha` del modelo. Ejecuta de
nuevo las dos celdas. Mira dónde acaba el percentil 90.

In [ ]:
with modelo_agrupado:
    idata_agrupado = pm.sample(
        draws=1000, tune=1000, chains=4, random_seed=SEMILLA, progressbar=False
    )

### OLS y bayesiano dan los mismos números

Con 919 datos, la priori pesa poco. La media de la posteriori cae sobre OLS. El bayesiano
añade la distribución completa de cada parámetro.

In [ ]:
resumen = az.summary(idata_agrupado, var_names=["alpha", "beta1", "sigma"])

pd.DataFrame({
    "OLS": [ols.params["const"], ols.params["floor"], np.sqrt(ols.scale)],
    "Bayesiano (media)": resumen["mean"].values,
    "HDI 3%": resumen["hdi_3%"].values,
    "HDI 97%": resumen["hdi_97%"].values,
}, index=["alpha / const", "beta1 / floor", "sigma"]).round(3)

### Pero los condados no son iguales

El Modelo 1 da un solo nivel para todo Minnesota. Miramos la media de cada condado.

In [ ]:
n_obs = np.bincount(county_idx)
media_condado = df.groupby("county_code")["log_radon"].mean().values
nombres = df.groupby("county_code")["county"].first().values

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(n_obs, media_condado, color="black", s=25, alpha=0.65)
ax.axhline(log_radon.mean(), color=ACENTO, linestyle="--", linewidth=1.5,
           label="Media de Minnesota")
ax.set_xscale("log")
ax.set_xlabel("Mediciones en el condado")
ax.set_ylabel("Media de log(radón)")
ax.legend()
plt.show()

print(f"{(n_obs <= 3).sum()} condados tienen 3 mediciones o menos. ST LOUIS tiene {n_obs.max()}.")

Las medias van de 0,4 a 2,6. Los condados son distintos. Un único $\alpha$ no los recoge.

Los puntos más dispersos están a la izquierda. Son condados con muy pocas mediciones.
Esto va a dar problemas.

## 3. Modelo 2: un nivel por condado

Cada condado tiene su $\alpha$. Son 85. La pendiente y $\sigma$ se comparten. En el código
cambia una cosa: `shape=n_condados`.

In [ ]:
with pm.Model() as modelo_unpooled:
    alpha = pm.Normal("alpha", mu=0, sigma=2, shape=n_condados)  # un alpha por condado
    beta1 = pm.Normal("beta1", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    y_obs = pm.Normal(
        "y_obs", mu=alpha[county_idx] + beta1 * floor, sigma=sigma, observed=log_radon
    )

    previa_unpooled = pm.sample_prior_predictive(draws=500, random_seed=SEMILLA)

pintar_previa(previa_unpooled)

La previa sale parecida a la del Modelo 1. Ajustamos.

In [ ]:
with modelo_unpooled:
    idata_unpooled = pm.sample(
        draws=1000, tune=1000, chains=4, random_seed=SEMILLA, progressbar=False
    )

### Los niveles extremos son de condados con poca muestra

Cada punto es un condado. El eje vertical es la media de su $\alpha$ en la posteriori.
Etiquetamos los dos más bajos y los dos más altos, con sus mediciones entre paréntesis.

In [ ]:
alpha_unpooled = idata_unpooled.posterior["alpha"].mean(("chain", "draw")).values
extremos = np.argsort(alpha_unpooled)[[0, 1, -2, -1]]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(n_obs, alpha_unpooled, color="black", s=25, alpha=0.65)
for c in extremos:
    ax.annotate(f"{nombres[c]} ({n_obs[c]})", (n_obs[c], alpha_unpooled[c]),
                xytext=(6, 0), textcoords="offset points", va="center",
                fontsize=10, color=ACENTO)
ax.set_xscale("log")
ax.set_xlabel("Mediciones en el condado")
ax.set_ylabel("α estimado (Modelo 2)")
plt.show()

LAC QUI PARLE sale el condado con más radón de Minnesota. Tiene dos casas medidas.
WATONWAN sale segundo, con tres. Por abajo, COOK tiene dos.

El modelo se cree esas pocas casas. Si midieron alto, el condado entero sale alto. Los
otros 84 condados saben qué niveles son plausibles. Este modelo no los mira.

## 4. Modelo 3: los condados se prestan información

Los 85 niveles salen de una distribución común. Su media y su dispersión se estiman.

$$
\begin{aligned}
y_i &\sim \mathcal{N}(\alpha_{c[i]} + \beta_1 x_i, \; \sigma^2) \\[4pt]
\alpha_c &\sim \mathcal{N}(\mu_\alpha, \sigma_\alpha^2) \\[4pt]
\mu_\alpha &\sim \mathcal{N}(0, 2^2), \qquad \sigma_\alpha \sim \text{Exp}(1)
\end{aligned}
$$

Respecto al Modelo 2 cambia una línea.

Su previa también está comprobada. Sale como la del Modelo 2, con la cola algo más larga.
Aquí no la repetimos.

In [ ]:
with pm.Model() as modelo_jerarquico:
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=2)
    sigma_alpha = pm.Exponential("sigma_alpha", 1)

    alpha = pm.Normal("alpha", mu=mu_alpha, sigma=sigma_alpha, shape=n_condados)  # <—
    beta1 = pm.Normal("beta1", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    y_obs = pm.Normal(
        "y_obs", mu=alpha[county_idx] + beta1 * floor, sigma=sigma, observed=log_radon
    )

    idata_jerarquico = pm.sample(
        draws=1000, tune=1000, chains=4, target_accept=0.95,
        random_seed=SEMILLA, progressbar=False,
    )

Los tres modelos son el mismo. Cambia la restricción sobre los $\alpha_c$:

- **Modelo 1**: todos iguales. Equivale a $\sigma_\alpha = 0$.
- **Modelo 2**: sin relación entre ellos. Equivale a $\sigma_\alpha = \infty$.
- **Modelo 3**: $\sigma_\alpha$ se estima con los datos.

> **Nota.** Aquí no hace falta la parametrización no centrada. El modelo centrado muestrea
> limpio con `target_accept=0.95`. Lo comprobamos en la sección 5.

### Los condados con pocos datos se acercan a la media

El mismo gráfico de antes. Punto negro: Modelo 2. Círculo morado: Modelo 3. La línea gris
une los dos.

In [ ]:
alpha_jer = idata_jerarquico.posterior["alpha"].mean(("chain", "draw")).values
mu_global = float(idata_jerarquico.posterior["mu_alpha"].mean())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.vlines(n_obs, alpha_unpooled, alpha_jer, color="#BBBBBB", linewidth=0.8, zorder=0)
ax.scatter(n_obs, alpha_unpooled, color="black", s=25, alpha=0.65, label="Modelo 2")
ax.scatter(n_obs, alpha_jer, facecolors="white", edgecolors=ACENTO, s=30,
           linewidths=1.3, label="Modelo 3")
for c in extremos:
    ax.annotate(nombres[c], (n_obs[c], alpha_unpooled[c]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=10, color=ACENTO)
ax.axhline(mu_global, color="black", linestyle="--", linewidth=1, label="Media global")
ax.set_xscale("log")
ax.set_xlabel("Mediciones en el condado")
ax.set_ylabel("α estimado")
ax.legend()
plt.show()

A la izquierda, líneas largas. LAC QUI PARLE baja de 2,7 a 1,9. Los condados con pocas
mediciones se acercan a Minnesota. A la derecha, casi no hay líneas. Con cien mediciones
el condado se sostiene solo.

El tamaño del tirón sale de dos varianzas del modelo. Una mide cuánto difieren los
condados. La otra, cuánto difieren las casas de un mismo condado.

In [ ]:
s_alpha = float(idata_jerarquico.posterior["sigma_alpha"].mean())
s_resid = float(idata_jerarquico.posterior["sigma"].mean())
n_equivalente = (s_resid / s_alpha) ** 2

print(f"σ_alpha (entre condados)   = {s_alpha:.2f}")
print(f"σ       (entre viviendas)  = {s_resid:.2f}")
print(f"Lo que el modelo sabe de Minnesota pesa como {n_equivalente:.0f} mediciones propias.")

Por debajo de ese número manda Minnesota. Por encima manda el condado. Nadie ha elegido
el umbral. Sale de los datos.

## 5. Diagnóstico: ¿ha funcionado el muestreador?

Antes de leer la posteriori se comprueba el muestreo. Primero, la traza. Las cuatro cadenas
deben solaparse.

In [ ]:
az.plot_trace(
    idata_jerarquico,
    var_names=["mu_alpha", "sigma_alpha", "beta1", "sigma"],
    chain_prop={"color": GRISES, "linestyle": ["-", "--", ":", "-."]},
    figsize=(9, 7),
)
plt.tight_layout()
plt.show()

Después, los números. `r_hat` debe rondar 1. El ESS cuenta muestras efectivas; bastan unos
cientos. Las divergencias deben ser cero.

In [ ]:
resumen_jer = az.summary(
    idata_jerarquico, var_names=["mu_alpha", "sigma_alpha", "beta1", "sigma"]
)
n_divergencias = int(idata_jerarquico.sample_stats["diverging"].sum())

print(f"Divergencias: {n_divergencias}")
resumen_jer[["mean", "hdi_3%", "hdi_97%", "r_hat", "ess_bulk", "ess_tail"]]

Todo en orden. Falta ver si el modelo reproduce los datos.

In [ ]:
with modelo_jerarquico:
    pm.sample_posterior_predictive(
        idata_jerarquico, extend_inferencedata=True,
        random_seed=SEMILLA, progressbar=False,
    )

ax = az.plot_ppc(
    idata_jerarquico, num_pp_samples=100, random_seed=SEMILLA,
    colors=["#AAAAAA", "black", ACENTO], figsize=(8, 4),
)
for linea in ax.get_lines():
    linea.set_linestyle("-")
ax.set_xlabel("log(radón)")
ax.set_ylabel("Densidad")
ax.legend(fontsize=10)
plt.show()

Las réplicas cubren los datos. Se les escapa un pico cerca de 1. El modelo es normal y los
datos, un poco menos.

## 6. Para qué sirve

La pregunta era sobre una casa. Hay dos respuestas:

- El **nivel del condado**.
- El **nivel de una vivienda**: el del condado más la variación entre casas, $\sigma$.

Comparamos ST LOUIS, con 116 mediciones, y LAC QUI PARLE, con 2. Las dos, midiendo en el
sótano.

In [ ]:
post = idata_jerarquico.posterior
s = post["sigma"].values.reshape(-1)


def prediccion(condado):
    # Intervalos al 94 % en pCi/L, para una medición en el sótano (planta = 0).
    c = list(nombres).index(condado)
    mu = post["alpha"].values[:, :, c].reshape(-1)
    y = rng.normal(mu[:, None], s[:, None], size=(len(mu), 20)).reshape(-1)  # 20 casas por muestra

    hdi_condado = np.exp(az.hdi(mu, hdi_prob=0.94))
    hdi_casa = np.exp(az.hdi(y, hdi_prob=0.94))
    return {
        "Condado 3%": hdi_condado[0],
        "Condado 97%": hdi_condado[1],
        "Casa 3%": hdi_casa[0],
        "Casa 97%": hdi_casa[1],
        "P(casa > 4 pCi/L)": (np.exp(y) > 4).mean(),
    }


condados = ["ST LOUIS", "LAC QUI PARLE"]
pd.DataFrame(
    [prediccion(c) for c in condados],
    index=[f"{c} ({(df['county'] == c).sum()} mediciones)" for c in condados],
).round(2)

El intervalo del condado se ensancha en LAC QUI PARLE. De él sabemos poco.

El de la casa es ancho en los dos. Su extremo alto multiplica por 15 o más al bajo. Lo
domina $\sigma$, la variación entre casas. Más mediciones en el condado no lo estrechan.

Conocer el condado no basta. Cada casa se mide.

## Lo que te llevas

- Juntar y separar son casos extremos del jerárquico. El jerárquico estima cuánto se
  parecen los condados.
- La priori se simula antes de ajustar.
- El muestreo se diagnostica antes de leer la posteriori.
- La respuesta es una distribución.

**Para seguir.** Gelman y Hill, *Data Analysis Using Regression and Multilevel/Hierarchical
Models*, de donde salen estos datos. Y McElreath, *Statistical Rethinking*, con sus clases
en YouTube.

---

Esto es el regalo de suscripción de mi newsletter. Escribo sobre estadística y análisis de
datos con más sarcasmo del recomendable: **[leonardohansa.com](https://leonardohansa.com)**.

Las slides del seminario están en
**[lhansa.github.io/seminar-bayes](https://lhansa.github.io/seminar-bayes/)**.

Si ejecutas esto, lo rompes o lo mejoras, cuéntamelo respondiendo a cualquier correo. Los
leo todos.